In [ ]:
import pandas as pd
import numpy as np
from astropy.table import Table
import matplotlib.pyplot as plt
from importlib import reload
import time
from pathlib import Path

import os

from importlib import reload

from IPython.display import display, HTML


def pprint(pandas_dataframe: pd.DataFrame, max_width=1000, **kwargs):
    return HTML(
        "\n".join(
            Table.from_pandas(pandas_dataframe).pformat(max_width=max_width, **kwargs)
        )
    )


import class_analytics_utils as cau

reload(cau)
pd.set_option("display.max_colwidth", None)

from nlp_summary import init_nltk
# init_nltk()

USE_FRESH_DATA = True
OVERWRITE = True

## Data Loading and Setup
- Loads survey data from local CSV or API.
- Cleans up question descriptions and aligns columns between pre and post.
- Drops unnecessary columns.
- Uses pandas DataFrames for all tabular data.

In [ ]:
# download all surveys
reload(cau)
import plo1 as plo1
reload(plo1)
reload(cau)
# from qualtrics_keys import (
#     survey_id_2025_post,
#     survey_id_2025_pre,
#     survey_id_2024_post,
#     survey_id_2024_pre,
# )

# import all keys from qualtrics_keys
from qualtrics_keys import (
    survey_id_dict
    )

id_column = "Intro information_1"


for which_survey in ['2023_fall']: #survey_id_dict.keys():

    # pre_response = cau.load_pre_data(from_api=False)
    file_exists = os.path.exists(survey_id_dict[which_survey]["output_name"]+'.csv')

    pre_response = cau.load_data(
        from_api=USE_FRESH_DATA,
        survey_id=survey_id_dict[which_survey]["pre"],
        # filename="2025_pre_response.csv",
        filename=Path("./surveys") / (survey_id_dict[which_survey]["output_name"]+'_pre.csv'),
    )

    post_response = cau.load_data(
        from_api=USE_FRESH_DATA,
        survey_id=survey_id_dict[which_survey]["post"],
        # filename="2025_post_response.csv",
        filename=Path("./surveys") / (survey_id_dict[which_survey]["output_name"]+'_post.csv'),
    )
        
        
    header_pre, description_pre, data_pre = cau.parse_response(pre_response)

    header_post, description_post, data_post = cau.parse_response(post_response)


    # updates some columns to match properly, and drop columns in the list
        # updates some columns to match properly, and drop columns in the list
    drop_columns = [
        "CosmicDS Pre-Survey - Click to write Form Field 4",
        "The CosmicDS team has my permission to use my anonymized survey data in their evaluation process."
        ]
    cau.column_cleanup(
        header_pre,
        description_pre,
        data_pre,
        header_post,
        description_post,
        data_post,
        drop_columns=drop_columns,
    )

    for question in description_post:
        # makes question tags match between matching questions
        # probably don't need to do this since we do everything based on the question text
        cau.get_pre_header_for_post_question(
            question,
            header_pre,
            description_pre,
            header_post,
            description_post,
            verbose=False,
        )
        
    plo1.process_survey_data(
    data_pre, description_pre, header_pre,
    data_post, description_post, header_post,
    combined_questions_filename = (which_survey+"_combined_questions.xlsx"),
    retrospective_full_filename = 
        str(Path("./retrospectives") / (which_survey+"_retrospective_full_results.xlsx"))
)




    # print out the number of records
    print("\n\n\n Number of records in Pre/Post survey")
    print("Pre-survey records: ", len(data_pre))
    print("Post-survey records: ", len(data_post))

In [ ]:
which_survey

In [ ]:
import plo1 as plo1
reload(plo1)
reload(cau)


plo1.process_survey_data(
    data_pre, description_pre, header_pre,
    data_post, description_post, header_post,
    combined_questions_filename = (which_survey+"_combined_questions.xlsx"),
    retrospective_full_filename = 
        str(Path("./retrospectives") / (which_survey+"_retrospective_full_results.xlsx"))
)
